# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates a workflow for loading, exploring, and analyzing the FAIR² human protein dataset using the `mlcroissant` library.

### Dataset Source
Croissant schema [available here](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

Dataset description: This dataset details the analysis of protein abundance, modifications, and sequences in human (Homo sapiens) samples. It includes fields such as accession, description, coverage percentage, peptide counts, MW, pI, and normalized abundances across samples.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {getattr(metadata, 'name', '')}\n\n{getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available **record sets** and **fields**. All entities are referenced by their `@id` as per Croissant conventions.

In [ ]:
# List all record sets and their fields
record_sets = list(metadata.record_sets)
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet name: {getattr(rs, 'name', '')}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id}, data_type: {field.data_type})")
    print("\n---\n")

## 3. Data Extraction
Load records from a specific record set using `@id`. (Note: If multiple record sets, you can extract each similarly.)

In [ ]:
# Collect all record set @id's
record_set_ids = [rs.id for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records_iter = dataset.records(record_set=record_set_id)
    df = pd.DataFrame(records_iter)
    dataframes[record_set_id] = df
    print(f"  Columns: {list(df.columns)}\n  {df.shape[0]} rows.\n---\n")

# Choose first record set for exploration
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    main_df = dataframes[main_record_set_id]
    print(f"First few records for RecordSet: {main_record_set_id}")
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Demonstrate typical data operations using column `@id`, e.g., normalization, filtering, grouping.

**Note:** The exact column `@id`s can be observed in the printout above. You can also inspect fields in the data overview section.

In [ ]:
# Example: filter & normalize a numeric field
# Please adapt numeric_field_id to your desired field @id, e.g.:
# Example ids (replace with the actual @id from your dataset):
numeric_field_id = None
group_field_id = None
# Get numeric columns
numeric_cols = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
if numeric_cols:
    numeric_field_id = numeric_cols[0]
    print(f"Using field id for numeric analysis: {numeric_field_id}")
    # Example filter: greater than a threshold
    threshold = 10
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold} (rows: {len(filtered_df)}):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Sample normalized {numeric_field_id} values:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by a categorical field
    cat_cols = [col for col in main_df.columns if pd.api.types.is_string_dtype(main_df[col]) and col != numeric_field_id]
    if cat_cols:
        group_field_id = cat_cols[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships. (You may customize with your field `@id`s).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If a second numeric column exists, show scatter
    if len(numeric_cols) > 1:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=main_df[numeric_cols[0]], y=main_df[numeric_cols[1]])
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.title(f'Scatter: {numeric_cols[0]} vs {numeric_cols[1]}')
        plt.show()


## 6. Conclusion
 - We demonstrated how to load and inspect a FAIR² Croissant-structured dataset with `mlcroissant`.
 - All references to record sets, fields, and columns use their Croissant `@id` for robust code.
 - Basic EDA and visualization showcased the utility of the dataset for further scientific and ML tasks.

> Please consult dataset documentation and field `@id` mapping for domain-specific analysis. Happy exploring!